In [1]:
!pip install transformers datasets evaluate rouge-score nltk --q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from datasets import load_dataset
from evaluate import load

model_checkpoint = "t5-small"

raw_datasets = load_dataset("EdinburghNLP/xsum")
metric = load("rouge")

README.md:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  300MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.7MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [3]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [4]:
raw_datasets["train"][1]

{'document': 'A fire alarm went off at the Holiday Inn in Hope Street at about 04:20 BST on Saturday and guests were asked to leave the hotel.\nAs they gathered outside they saw the two buses, parked side-by-side in the car park, engulfed by flames.\nOne of the tour groups is from Germany, the other from China and Taiwan. It was their first night in Northern Ireland.\nThe driver of one of the buses said many of the passengers had left personal belongings on board and these had been destroyed.\nBoth groups have organised replacement coaches and will begin their tour of the north coast later than they had planned.\nPolice have appealed for information about the attack.\nInsp David Gibson said: "It appears as though the fire started under one of the buses before spreading to the second.\n"While the exact cause is still under investigation, it is thought that the fire was started deliberately."',
 'summary': 'Two tourist buses have been destroyed by fire in a suspected arson attack in Belf

In [5]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=5):
  assert num_examples < len(dataset), "num_examples can't be smaller then length of dataset"
  picks = []
  for num in range(num_examples):
    pick = random.randint(0, len(dataset)-1)
    while pick in picks:
      pick = random.randint(0, len(dataset)-1)
    picks.append(pick)

  df = pd.DataFrame(dataset[picks])
  for column, typ in dataset.features.items():
    if isinstance(typ, datasets.ClassLabel):
      df[column] = df[column].transform(lambda i: typ.names[i])
  display(HTML(df.to_html()))

In [6]:
print(len(raw_datasets["train"]))

204045


In [7]:
show_random_elements(raw_datasets["train"])

In [8]:
fake_preds = ["hello there", "general kenobi"]
fake_labels = ["hello there", "general kenobi"]

#how to use ROUGE
metric.compute(predictions=fake_preds, references=fake_labels)

{'rouge1': np.float64(1.0),
 'rouge2': np.float64(1.0),
 'rougeL': np.float64(1.0),
 'rougeLsum': np.float64(1.0)}

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [10]:
d1_token = tokenizer("hello, this is a test sentence")
d2_token = tokenizer(["hello, this is a test sentence","this is another sentence"])
print(d1_token)
print(d2_token)

{'input_ids': [21820, 6, 48, 19, 3, 9, 794, 7142, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
{'input_ids': [[21820, 6, 48, 19, 3, 9, 794, 7142, 1], [48, 19, 430, 7142, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]}


In [11]:
#target token makes sure we use special tokens

print(tokenizer(text_target=["hello, this is a sentence"]))


{'input_ids': [[21820, 6, 48, 19, 3, 9, 7142, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1]]}


In [12]:
if model_checkpoint in ['t5-small','t5-base','t5-large','t5-3b','t5-11b']:
  prefix = "summarize:"
else:
  prefix = ""

In [13]:
def preprocess_function(example):
  inputs = [prefix + doc for doc in example["document"]]
  input_ids = tokenizer(inputs, max_length = 1024, truncation=True)

  labels =  tokenizer(text_target=example['summary'], max_length = 128, truncation=True)

  input_ids["labels"] = labels["input_ids"]
  return input_ids

In [14]:
print(preprocess_function(raw_datasets['train'][:2]))

{'input_ids': [[21603, 10, 634, 423, 583, 13, 1783, 16, 20126, 16496, 6, 80, 13, 8, 844, 6025, 4161, 6, 19, 341, 271, 14841, 5, 7057, 161, 19, 4912, 16, 1626, 5981, 11, 186, 7540, 16, 1276, 15, 2296, 7, 5718, 2367, 14621, 4161, 57, 4125, 387, 5, 15059, 7, 30, 8, 4653, 4939, 711, 747, 522, 17879, 788, 12, 1783, 44, 8, 15763, 6029, 1813, 9, 7472, 5, 1404, 1623, 11, 5699, 277, 130, 4161, 57, 18368, 16, 20126, 16496, 227, 8, 2473, 5895, 15, 147, 89, 22411, 139, 8, 1511, 5, 1485, 3271, 3, 21926, 9, 472, 19623, 5251, 8, 616, 12, 15614, 8, 1783, 5, 37, 13818, 10564, 15, 26, 3, 9, 3, 19513, 1481, 6, 18368, 186, 1328, 2605, 30, 7488, 1887, 3, 18, 8, 711, 2309, 9517, 89, 355, 5, 3966, 1954, 9233, 15, 6, 113, 293, 7, 8, 16548, 13363, 106, 14022, 84, 47, 14621, 4161, 6, 243, 255, 228, 59, 7828, 8, 1249, 18, 545, 11298, 1773, 728, 8, 8347, 1560, 5, 611, 6, 255, 243, 72, 1709, 1528, 161, 228, 43, 118, 4006, 91, 12, 766, 8, 3, 19513, 1481, 410, 59, 5124, 5, 96, 196, 17, 19, 1256, 68, 27, 103, 317, 13

In [15]:
#better to use map when applying a function on a dataset

tokenized_dataset = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/204045 [00:00<?, ? examples/s]

Map:   0%|          | 0/11332 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [16]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,  Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
batch_size = 16

model_name = model_checkpoint
args = Seq2SeqTrainingArguments(
    f"{model_name}-finetuned-xsum",
    learning_rate = 2e-5,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    weight_decay = 0.01,
    save_total_limit = 3,
    num_train_epochs= 1,
    predict_with_generate = True,
    fp16 = True,
    report_to = "none"
)

In [18]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [19]:
import nltk
import numpy as np

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

  labels = np.where(labels != -100, labels, skip_special_tokens=True)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in predictions]
  decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in labels]

  result = metric.compute(predictions=decoded_preds, reference = decoded_labels)
  result = {key: value * 100 for key ,value in result.items()}

  prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
  result["gen_len"] = np.mean(prediction_lens)

  return {k: round(v, 4) for k, v in result.items()}

In [20]:
#??Seq2SeqTrainer

In [21]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset= tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    data_collator = data_collator,
    #tokenizer = tokenizer,
    compute_metrics = compute_metrics
)

In [22]:
trainer.train()

Step,Training Loss
500,3.056998
1000,2.888307
1500,2.851012
2000,2.838472
2500,2.812515
3000,2.800991
3500,2.781988
4000,2.786139
4500,2.761539
5000,2.760804


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (18 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12753, training_loss=2.771507538973673, metrics={'train_runtime': 9163.0029, 'train_samples_per_second': 22.268, 'train_steps_per_second': 1.392, 'total_flos': 5.403369058507162e+16, 'train_loss': 2.771507538973673, 'epoch': 1.0})